# Random Drop Testing & Visualization (ImageNet-100)

This notebook evaluates random token dropping across keep rates and visualizes retained vs dropped patches.

In [ ]:
from google.colab import userdata

token = userdata.get('GithubPAT')

!git clone https://{token}@github.com/Chalhotra/ViT-Token-Economy.git
%cd ViT-Token-Economy

In [ ]:
import gc
import torch
from matplotlib import pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import torchvision.transforms as T
import requests

In [ ]:
from src.test_models.random_drop import PrunedViT

In [ ]:
!pip -q install -r requirements.txt
!pip -q install -e .

In [ ]:
from src.imagenet_mapping import build_imagenet100_to_1k_map
from src.models import ModelConfig, create_model, shrink_imagenet1k_head_to_imagenet100
from src.data import DataConfig, load_imagenet100_split, build_transform_for_model, apply_timm_preprocess, build_loader
from src.eval import evaluate_accuracy_latency_throughput, compute_gflops
from src.utils import get_device, num_params

In [ ]:
device = get_device()
maps = build_imagenet100_to_1k_map()
ds = load_imagenet100_split(DataConfig(split='validation'))

In [ ]:
def run_pruned_vit(model_id: str, ds = ds, batch_size: int = 64, prune_layers: tuple = (2, 5, 8), keep_ratios: tuple = (0.70, 0.5, 0.25)):
    base_model_name = model_id.replace('pruned_', '')
    model = PrunedViT(model_name=base_model_name, prune_layers=prune_layers, keep_ratios=keep_ratios)
    shrink_imagenet1k_head_to_imagenet100(model.model, maps.new_to_old_map, num_classes=100)
    model = model.to(device).eval()
    transform = build_transform_for_model(model)
    ds_t = apply_timm_preprocess(ds, transform)
    loader = build_loader(ds_t, DataConfig(batch_size=batch_size, split='validation', shuffle=False))
    metrics = evaluate_accuracy_latency_throughput(model, loader, device)
    sample = ds_t[0]['pixel_values'].unsqueeze(0).to(device)
    gflops = compute_gflops(model, sample)
    return {
        'model': model_id,
        'params_m': num_params(model)/1e6,
        'gflops': gflops,
        **metrics
    }

def run(model_id: str, ds = ds, batch_size: int = 64, prune_layers: tuple = (2, 5, 8), keep_ratios: tuple = (0.70, 0.5, 0.25)):
    if model_id.startswith('pruned_'):
        return run_pruned_vit(model_id, ds, batch_size, prune_layers, keep_ratios)
    else:
        # Create a regular timm model
        model = create_model(ModelConfig(model_id=model_id, pretrained=True))
        # Shrink the head of the regular timm model directly
        shrink_imagenet1k_head_to_imagenet100(model, maps.new_to_old_map, num_classes=100)

    model = model.to(device).eval()
    transform = build_transform_for_model(model) # This uses model.pretrained_cfg if available
    ds_t = apply_timm_preprocess(ds, transform)
    loader = build_loader(ds_t, DataConfig(batch_size=batch_size, split='validation', shuffle=False))
    metrics = evaluate_accuracy_latency_throughput(model, loader, device)
    sample = ds_t[0]['pixel_values'].unsqueeze(0).to(device)
    gflops = compute_gflops(model, sample)
    return {
        'model': model_id,
        'params_m': num_params(model)/1e6,
        'gflops': gflops,
        **metrics
    }



In [ ]:
models = [
    'pruned_vit_tiny_patch16_224',
    'pruned_vit_small_patch16_224',
    'pruned_vit_base_patch16_224',
    'pruned_deit_tiny_patch16_224',
    'pruned_deit_small_patch16_224',
    'pruned_deit_base_patch16_224'
]

results_25 = {}

for m in models:
    print(f"Running {m}...")
    results_25[m] = run(m,prune_layers=(2,5,8), keep_ratios=(0.25, 0.25, 0.25))
    print(f"Finished {m}")

    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
from IPython.display import display, Markdown
import pandas as pd
df = pd.DataFrame(results_25).T
display(Markdown("## Vision Random Drop Experiment Results (keep rate - 25 % in each step)"))
display(df.drop('model',axis = 'columns'))

In [ ]:
models = [
    'pruned_vit_tiny_patch16_224',
    'pruned_vit_small_patch16_224',
    'pruned_vit_base_patch16_224',
    'pruned_deit_tiny_patch16_224',
    'pruned_deit_small_patch16_224',
    'pruned_deit_base_patch16_224'
]

results_50 = {}

for m in models:
    print(f"Running {m}...")
    results_50[m] = run(m,prune_layers=(2,5,8), keep_ratios=(0.50, 0.50, 0.50))
    print(f"Finished {m}")

    gc.collect()
    torch.cuda.empty_cache()

df = pd.DataFrame(results_50).T
display(Markdown("## Vision Random Drop Experiment Results (keep rate - 50 % in each step)"))
display(df.drop('model',axis = 'columns'))

In [ ]:
models = [
    'pruned_vit_tiny_patch16_224',
    'pruned_vit_small_patch16_224',
    'pruned_vit_base_patch16_224',
    'pruned_deit_tiny_patch16_224',
    'pruned_deit_small_patch16_224',
    'pruned_deit_base_patch16_224'
]

results_70 = {}

for m in models:
    print(f"Running {m}...")
    results_70[m] = run(m,prune_layers=(2,5,8), keep_ratios=(0.70, 0.70, 0.70))
    print(f"Finished {m}")

    gc.collect()
    torch.cuda.empty_cache()

df = pd.DataFrame(results_70).T
display(Markdown("## Vision Random Drop Experiment Results (keep rate - 70 % in each step)"))
display(df.drop('model',axis = 'columns'))

In [ ]:
models = [
    'pruned_vit_tiny_patch16_224',
    'pruned_vit_small_patch16_224',
    'pruned_vit_base_patch16_224',
    'pruned_deit_tiny_patch16_224',
    'pruned_deit_small_patch16_224',
    'pruned_deit_base_patch16_224'
]

results_90 = {}

for m in models:
    print(f"Running {m}...")
    results_90[m] = run(m,prune_layers=(2,5,8), keep_ratios=(0.90, 0.90, 0.90))
    print(f"Finished {m}")

    gc.collect()
    torch.cuda.empty_cache()

df = pd.DataFrame(results_90).T
display(Markdown("## Vision Random Drop Experiment Results (keep rate - 90 % in each step)"))
display(df.drop('model',axis = 'columns'))

## Visualizations

In [ ]:
m = PrunedViT()

url = 'https://raw.githubusercontent.com/EliSchwartz/imagenet-sample-images/master/n09468604_valley.JPEG'
response = requests.get(url, stream=True)
img = Image.open(response.raw)

transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

img_tensor = transform(img)
print(f"Original image tensor shape: {img_tensor.shape}")

## 3. Apply Random Pruning and Inspect Output

Here, we will explicitly call the patch embedding, add class and positional tokens, and then apply the `random_prune` method to see the effect on the token sequence shape.

In [ ]:
embedded_patches = m.model.patch_embed(img_tensor.unsqueeze(0))

cls_token = m.model.cls_token.expand(embedded_patches.shape[0], -1, -1)
tokens_with_cls = torch.cat((cls_token, embedded_patches), dim=1)
tokens_with_pos = tokens_with_cls + m.model.pos_embed
tokens_with_pos = m.model.pos_drop(tokens_with_pos)

keep_ratio_for_vis = 0.25

num_current_patches_before_pruning = tokens_with_pos.shape[1] - 1
target_k_for_vis = int(num_current_patches_before_pruning * keep_ratio_for_vis)

pruned_output_tokens = m.random_prune(tokens_with_pos, target_k_for_vis)

print(f"Shape after random_prune: {pruned_output_tokens.shape}")

## 4. Visualize Kept Patches

To understand which parts of the image are retained, we will denormalize the original image and highlight the patches that correspond to the tokens kept by the `random_prune` method.

In [ ]:
# Denormalize img_tensor for visualization
mean = torch.tensor([0.485, 0.456, 0.406]).to(img_tensor.device).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).to(img_tensor.device).view(3, 1, 1)

img_display = img_tensor * std + mean
img_display = torch.clamp(img_display, 0, 1)
img_display = img_display.permute(1, 2, 0).cpu().numpy()

print("Image tensor successfully denormalized and prepared for display.")

# Replicate patch selection logic to get indices of kept patches
# tokens_with_pos is (B, N, C), where N includes the cls token
cls_token_vis = tokens_with_pos[:, :1, :]
patch_tokens_vis = tokens_with_pos[:, 1:, :]

B, num_patches, C = patch_tokens_vis.shape
k = int(num_patches * keep_ratio_for_vis)

# Generate random indices, mirroring the random_prune method
# For visualization, we use the same `keep_ratio_for_vis`
idx = torch.rand(B, num_patches, device=patch_tokens_vis.device).argsort(dim=1)
kept_patch_indices = idx[:, :k]

print(f"Original number of patch tokens: {num_patches}")
print(f"Number of patch tokens to keep: {k}")
print(f"Shape of kept_patch_indices: {kept_patch_indices.shape}")

# Visualize the kept patches on the original image
img_size = 224 # vit_tiny_patch16_224 implies 224x224 input
patch_size = 16 # vit_tiny_patch16_224 implies 16x16 patches
num_patches_per_side = img_size // patch_size # 224 // 16 = 14

# Get kept and dropped indices
kept_indices_flat = set(kept_patch_indices[0].cpu().numpy())
all_patch_indices_flat = set(range(num_patches))
dropped_indices_flat = list(all_patch_indices_flat - kept_indices_flat)

fig, ax = plt.subplots(1, figsize=(4, 4))
ax.imshow(img_display)

# Draw dropped patches as filled black rectangles
for idx in dropped_indices_flat:
    row = idx // num_patches_per_side
    col = idx % num_patches_per_side
    x = col * patch_size
    y = row * patch_size
    rect = patches.Rectangle(
        (x, y),
        patch_size,
        patch_size,
        linewidth=0,
        edgecolor='none',
        facecolor='black'
    )
    ax.add_patch(rect)

ax.set_title(f'Original Image with Dropped Patches (Keep Ratio: {keep_ratio_for_vis})')
ax.axis('off')
plt.show()

print("Visualization of kept and dropped patches on the original image completed.")

In [ ]:
def visualize_random_pruning(img_tensor, model, keep_ratios=[0.9, 0.75, 0.5, 0.25]):

    device = img_tensor.device

    embedded_patches = model.model.patch_embed(img_tensor.unsqueeze(0))

    cls_token = model.model.cls_token.expand(embedded_patches.shape[0], -1, -1)
    tokens_with_cls = torch.cat((cls_token, embedded_patches), dim=1)

    tokens_with_pos = tokens_with_cls + model.model.pos_embed
    tokens_with_pos = model.model.pos_drop(tokens_with_pos)

    patch_tokens = tokens_with_pos[:, 1:, :]
    B, num_patches, C = patch_tokens.shape

    mean = torch.tensor([0.485, 0.456, 0.406]).to(device).view(3,1,1)
    std = torch.tensor([0.229, 0.224, 0.225]).to(device).view(3,1,1)

    img_display = img_tensor * std + mean
    img_display = torch.clamp(img_display, 0, 1)
    img_display = img_display.permute(1,2,0).cpu().numpy()

    img_size = 224
    patch_size = 16
    num_patches_per_side = img_size // patch_size

    fig, axes = plt.subplots(1, len(keep_ratios), figsize=(3*len(keep_ratios),3))

    if len(keep_ratios) == 1:
        axes = [axes]

    for ax, keep_ratio in zip(axes, keep_ratios):

        k = int(num_patches * keep_ratio)

        idx = torch.rand(B, num_patches, device=device).argsort(dim=1)
        kept_patch_indices = idx[:, :k]

        kept_indices_flat = set(kept_patch_indices[0].cpu().numpy())
        all_patch_indices_flat = set(range(num_patches))
        dropped_indices_flat = list(all_patch_indices_flat - kept_indices_flat)

        ax.imshow(img_display)

        for patch_idx in dropped_indices_flat:
            row = patch_idx // num_patches_per_side
            col = patch_idx % num_patches_per_side

            x = col * patch_size
            y = row * patch_size

            rect = patches.Rectangle(
                (x, y),
                patch_size,
                patch_size,
                linewidth=0,
                facecolor='black'
            )
            ax.add_patch(rect)

        ax.set_title(f"Keep Ratio: {keep_ratio}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
visualize_random_pruning(
    img_tensor,
    m,
    keep_ratios=[1,0.90, 0.70, 0.50, 0.25]
)

## 5. Test the Full PrunedViT Model

Now, let's pass the preprocessed image tensor through the full `PrunedViT` model's `forward` method to observe the final output shape after all pruning layers.

In [ ]:
model_output = m(img_tensor.unsqueeze(0))

print(f"Shape of output after full PrunedViT model forward pass: {model_output.shape}")

## 6. Extract Predicted Class from Model Output

Now, we will determine the predicted class from the `model_output` tensor. This involves finding the index with the highest logit value and mapping it to an ImageNet class label.

In [ ]:
# Get the predicted class index
predicted_class_idx = torch.argmax(model_output, dim=1).item()

# Load ImageNet class labels (if not already loaded)
# This is a common way to get ImageNet labels for timm models
url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
response = requests.get(url)
labels = response.text.split("\n")

# Get the predicted class name
predicted_class_name = labels[predicted_class_idx]

print(f"Predicted class index: {predicted_class_idx}")
print(f"Predicted class name: {predicted_class_name}")